#Competencia Kaggle

#Modelos y Simulación I Udea2025-2

In [9]:
!wget --no-cache -O init.py -q https://raw.githubusercontent.com/rramosp/ai4eng.v1/main/content/init.py
import init; init.init(force_download=False); init.get_weblink()

Configuración de Kaggle y descarga de datos

In [10]:
!mv kaggle.json /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json

!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia
!unzip udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
!unzip -l udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip

mv: cannot stat 'kaggle.json': No such file or directory
udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
replace submission_example.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
Archive:  udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
  4716673  2025-09-16 01:46   submission_example.csv
 59185238  2025-09-16 01:46   test.csv
143732437  2025-09-16 01:46   train.csv
---------                     -------
207634348                     3 files


Carga de librerías

In [11]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

Carga de datasets

In [12]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

Train: (692500, 21)
Test: (296786, 20)


Identificar variable objetivo

In [13]:
target_candidates = list(set(train.columns) - set(test.columns))
variable_objetivo = target_candidates[0]
print(f"Variable objetivo: {variable_objetivo}")

Variable objetivo: RENDIMIENTO_GLOBAL


Features

In [14]:
# 1. FEATURES BÁSICAS
train['suma_indicadores'] = train[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].sum(axis=1)
test['suma_indicadores'] = test[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].sum(axis=1)

train['promedio_indicadores'] = train[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].mean(axis=1)
test['promedio_indicadores'] = test[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].mean(axis=1)

# 2. FEATURES DE INTERACCIÓN
train['producto_1_4'] = train['INDICADOR_1'] * train['INDICADOR_4']
test['producto_1_4'] = test['INDICADOR_1'] * test['INDICADOR_4']

train['ratio_3_2'] = train['INDICADOR_3'] / (train['INDICADOR_2'] + 0.001)
test['ratio_3_2'] = test['INDICADOR_3'] / (test['INDICADOR_2'] + 0.001)

# 3. FEATURES DE VARIABILIDAD
train['rango_indicadores'] = train[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].max(axis=1) - train[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].min(axis=1)
test['rango_indicadores'] = test[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].max(axis=1) - test[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].min(axis=1)

train['std_indicadores'] = train[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].std(axis=1)
test['std_indicadores'] = test[['INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']].std(axis=1)

# 4. FEATURES BINARIAS
train['alto_indicador_1'] = (train['INDICADOR_1'] > train['INDICADOR_1'].median()).astype(int)
test['alto_indicador_1'] = (test['INDICADOR_1'] > test['INDICADOR_1'].median()).astype(int)

train['todos_altos'] = ((train['INDICADOR_1'] > train['INDICADOR_1'].median()) &
                        (train['INDICADOR_2'] > train['INDICADOR_2'].median()) &
                        (train['INDICADOR_3'] > train['INDICADOR_3'].median()) &
                        (train['INDICADOR_4'] > train['INDICADOR_4'].median())).astype(int)
test['todos_altos'] = ((test['INDICADOR_1'] > test['INDICADOR_1'].median()) &
                      (test['INDICADOR_2'] > test['INDICADOR_2'].median()) &
                      (test['INDICADOR_3'] > test['INDICADOR_3'].median()) &
                      (test['INDICADOR_4'] > test['INDICADOR_4'].median())).astype(int)

# 5. CODIFICAR CATEGÓRICAS
categorical_cols = ['E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'F_ESTRATOVIVIENDA', 'F_EDUCACIONPADRE', 'F_EDUCACIONMADRE']

for col in categorical_cols:
    if col in train.columns:
        le = LabelEncoder()
        combined = pd.concat([train[col], test[col]], axis=0)
        le.fit(combined.astype(str))
        train[f'{col}_encoded'] = le.transform(train[col].astype(str))
        test[f'{col}_encoded'] = le.transform(test[col].astype(str))

Seleccion de Features

In [15]:
exclude_cols = [variable_objetivo]
feature_cols = [col for col in train.columns
                if col not in exclude_cols and train[col].dtype in [np.int64, np.float64]]


Preprocesamiento

In [16]:
X_train = train[feature_cols]
y_train = train[variable_objetivo]
X_test = test[feature_cols]

# Escalar datos para SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
modelo_svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42
)

# Validación cruzada
scores = cross_val_score(modelo_svm, X_train_scaled, y_train, cv=5)
print(f"📈 Cross-validation SVM: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

Entrenamiento y evaluación

In [ ]:
modelo_svm.fit(X_train_scaled, y_train)

RandomForestClassifier(max_depth=15, min_samples_leaf=2, min_samples_split=5,
                       n_estimators=200, n_jobs=-1, random_state=42)

Predicción final

In [ ]:
predicciones = modelo_svm.predict(X_test_scaled)

Problema con la columna 'id'

Cambio de nombre de variable y descarga del submission

In [ ]:
id_col = [col for col in test.columns if 'id' in col.lower()][0]

submission = pd.DataFrame({
    'ID': test[id_col],
    variable_objetivo: predicciones
})

submission.to_csv('submission.csv', index=False)
from google.colab import files
files.download('submission_svm.csv')

Columnas en test: ['ID', 'PERIODO_ACADEMICO', 'E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA', 'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE', 'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PRIVADO_LIBERTAD', 'E_PAGOMATRICULAPROPIO', 'F_TIENECOMPUTADOR', 'F_TIENEINTERNET.1', 'F_EDUCACIONMADRE', 'INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4', 'suma_indicadores', 'promedio_indicadores', 'E_PRGM_ACADEMICO_encoded', 'E_PRGM_DEPARTAMENTO_encoded', 'F_ESTRATOVIVIENDA_encoded']
Columnas que podrían ser ID: ['ID', 'E_VALORMATRICULAUNIVERSIDAD']
Usando columna: 'ID'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>